# 🤖 Modelado y Evaluación con Modelo Base

Este notebook implementa el entrenamiento, optimización y evaluación de modelos de clasificación multiclase.

**Objetivo:** Entrenar y comparar múltiples algoritmos para seleccionar el mejor modelo.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, UnivariateFeatureSelector, OneHotEncoder, StandardScaler, PCA
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from pyspark.sql import Window
from pyspark.sql import functions as F

import mlflow
from datetime import datetime

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## 2. Cargar el conjunto de `datos`

In [0]:
# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

SILVER_FULL = qname(silver_table)
print("Tabla Silver:", SILVER_FULL)

In [0]:
# Leer datos de la tabla Bronze
lpn_silver = spark.table(SILVER_FULL)

display(lpn_silver.limit(20))

In [0]:
# Dimensiones del dataset
print(f"Filas: {lpn_silver.count()}, Columnas: {len(lpn_silver.columns)}")

### 2.1 Division de datos en train y test

In [0]:
target_col = "LABEL_ZONE"
train_frac = 0.8

# Añadir columna aleatoria
df = lpn_silver.withColumn("rand", F.rand())

# Calcular percentil por clase
window = Window.partitionBy(target_col).orderBy("rand")
df = df.withColumn("row_number", F.row_number().over(window))
df = df.withColumn("count_per_class", F.count("*").over(Window.partitionBy(target_col)))
df = df.withColumn("frac", F.col("row_number") / F.col("count_per_class"))

# Split estratificado
train_df = df.filter(F.col("frac") <= train_frac).drop("rand", "row_number", "count_per_class", "frac")
test_df = df.filter(F.col("frac") > train_frac).drop("rand", "row_number", "count_per_class", "frac")

print(f"Train: {train_df.count()} filas")
print(f"Test:  {test_df.count()} filas")

## 3. Transformacion de variables seleccionadas

### 3.1 Normalización/Escalado

In [0]:
num_cols = ["EST_WT",
    "STD_CASE_QTY",
    "STD_PACK_QTY",
    "UNIT_PRICE",
    "CONS_PRTY_DATE_month",
    "CONS_PRTY_DATE_day",
    "CONS_PRTY_DATE_day_of_week",
    "RCVD_DATE_month",
    "RCVD_DATE_hour"]

In [0]:
num_vec = VectorAssembler(inputCols=num_cols, outputCol="num_vec", handleInvalid="keep")
scaler  = StandardScaler(inputCol="num_vec", outputCol="num_std", withMean=True, withStd=True)

### 3.2 One-Hot Encoding 


In [0]:
cat_cols = ["STORE_DEPT",
    "MV_SIZE_UOM",
    "PROD_TYPE",
    "MERCH_TYPE"]

In [0]:
# Index + OneHot por categórica; luego ensamblar con numéricas
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh", handleInvalid="keep", dropLast=False) for c in cat_cols]

### 3.3 Ensamblado final de features (numéricas escaladas + categóricas OHE)

In [0]:
final_inputs = ["num_std"] + [f"{c}_oh" for c in cat_cols]
features = VectorAssembler(inputCols=final_inputs, outputCol="features", handleInvalid="keep")

In [0]:
# ver columnas del VectorAssembler features
print(features.getInputCols())

### 3.4 Variable Objetivo

In [0]:
label_indexer = StringIndexer(inputCol="LABEL_ZONE", outputCol="LABEL_ZONE_idx", handleInvalid="keep")

### 3.4 Pipeline

In [0]:
selector_pipe = Pipeline(stages=[label_indexer] + indexers + encoders + [num_vec, scaler, features])
selector_model = selector_pipe.fit(train_df)

In [0]:
train_final_df = selector_model.transform(train_df)

In [0]:
test_final_df = selector_model.transform(test_df)

In [0]:
display(test_final_df.limit(5))

## 4. Entramiento y Evaluacion de Modelos

In [0]:
# Evaluadores
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="LABEL_ZONE_idx", predictionCol="prediction", metricName="f1")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="LABEL_ZONE_idx", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="LABEL_ZONE_idx", predictionCol="prediction", metricName="weightedRecall")

### 4.1 LogisticRegression

In [0]:
# LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="LABEL_ZONE_idx", maxIter=100, elasticNetParam=0.0)

In [0]:
run_name = f"LogisticRegression_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    
    # entrenamiento de modelo
    model = lr.fit(train_final_df)
    
    # Predicciones en train y test
    pred_train = model.transform(train_final_df)
    pred_test = model.transform(test_final_df)
    
    # Métricas train
    f1_train = evaluator_f1.evaluate(pred_train)
    precision_train = evaluator_precision.evaluate(pred_train)
    recall_train = evaluator_recall.evaluate(pred_train)
    
    # Métricas test
    f1_test = evaluator_f1.evaluate(pred_test)
    precision_test = evaluator_precision.evaluate(pred_test)
    recall_test = evaluator_recall.evaluate(pred_test)
    
    # Log de métricas
    mlflow.log_metric("f1_weighted_train", f1_train)
    mlflow.log_metric("precision_weighted_train", precision_train)
    mlflow.log_metric("recall_weighted_train", recall_train)
    mlflow.log_metric("f1_weighted_test", f1_test)
    mlflow.log_metric("precision_weighted_test", precision_test)
    mlflow.log_metric("recall_weighted_test", recall_test)

    # Log del modelo
    mlflow.spark.log_model(model, "modelo_lr")

    # Log de parámetros del modelo
    mlflow.log_param("maxIter", lr.getMaxIter())
    mlflow.log_param("elasticNetParam", lr.getElasticNetParam())
    mlflow.log_param("featuresCol", lr.getOrDefault("featuresCol"))
    mlflow.log_param("labelCol", lr.getOrDefault("labelCol"))

### 4.2 DecisionTreeClassifier

In [0]:
# DecisionTreeClassifier
dt = DecisionTreeClassifier(featuresCol="features", labelCol="LABEL_ZONE_idx", seed=42)

In [0]:
run_name = f"DecisionTreeClassifier_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    
    # entrenamiento de modelo
    model = dt.fit(train_final_df)
    
    # Predicciones en train y test
    pred_train = model.transform(train_final_df)
    pred_test = model.transform(test_final_df)
    
    # Métricas train
    f1_train = evaluator_f1.evaluate(pred_train)
    precision_train = evaluator_precision.evaluate(pred_train)
    recall_train = evaluator_recall.evaluate(pred_train)
    
    # Métricas test
    f1_test = evaluator_f1.evaluate(pred_test)
    precision_test = evaluator_precision.evaluate(pred_test)
    recall_test = evaluator_recall.evaluate(pred_test)
    
    # Log de métricas
    mlflow.log_metric("f1_weighted_train", f1_train)
    mlflow.log_metric("precision_weighted_train", precision_train)
    mlflow.log_metric("recall_weighted_train", recall_train)
    mlflow.log_metric("f1_weighted_test", f1_test)
    mlflow.log_metric("precision_weighted_test", precision_test)
    mlflow.log_metric("recall_weighted_test", recall_test)
    
    # Log del modelo
    mlflow.spark.log_model(model, "model_dt")
    
    # Log del experimento
    mlflow.log_param("max_depth", dt.getOrDefault("maxDepth"))
    mlflow.log_param("maxBins", dt.getOrDefault("maxBins"))
    mlflow.log_param("featuresCol", dt.getOrDefault("featuresCol"))
    mlflow.log_param("labelCol", dt.getOrDefault("labelCol"))

### 4.3 RandomForestClassifier

In [0]:
# RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol="LABEL_ZONE_idx", seed=42, numTrees=100)

In [0]:
run_name = f"RandomForest_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    
    # entrenamiento de modelo
    model = rf.fit(train_final_df)
    
    # Predicciones en train y test
    pred_train = model.transform(train_final_df)
    pred_test = model.transform(test_final_df)
    
    # Métricas train
    f1_train = evaluator_f1.evaluate(pred_train)
    precision_train = evaluator_precision.evaluate(pred_train)
    recall_train = evaluator_recall.evaluate(pred_train)
    
    # Métricas test
    f1_test = evaluator_f1.evaluate(pred_test)
    precision_test = evaluator_precision.evaluate(pred_test)
    recall_test = evaluator_recall.evaluate(pred_test)
    
    # Log de métricas
    mlflow.log_metric("f1_weighted_train", f1_train)
    mlflow.log_metric("precision_weighted_train", precision_train)
    mlflow.log_metric("recall_weighted_train", recall_train)
    mlflow.log_metric("f1_weighted_test", f1_test)
    mlflow.log_metric("precision_weighted_test", precision_test)
    mlflow.log_metric("recall_weighted_test", recall_test)

    # Log del modelo
    mlflow.spark.log_model(model, "modelo_rf")

    # Log de parámetros del modelo
    mlflow.log_param("numTrees", rf.getNumTrees())
    mlflow.log_param("maxDepth", rf.getOrDefault("maxDepth"))
    mlflow.log_param("featuresCol", rf.getOrDefault("featuresCol"))
    mlflow.log_param("labelCol", rf.getOrDefault("labelCol"))

### 4.4 XGBoost

In [0]:
# xgboost
xgb = SparkXGBClassifier(features_col="features", label_col="LABEL_ZONE_idx", enable_sparse_data_optim=True, missing=0.0)

In [0]:
run_name = f"XGBoost_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    
    # entrenamiento de modelo
    model = xgb.fit(train_final_df)
    
    # Predicciones en train y test
    pred_train = model.transform(train_final_df)
    pred_test = model.transform(test_final_df)
    
    # Métricas train
    f1_train = evaluator_f1.evaluate(pred_train)
    precision_train = evaluator_precision.evaluate(pred_train)
    recall_train = evaluator_recall.evaluate(pred_train)
    
    # Métricas test
    f1_test = evaluator_f1.evaluate(pred_test)
    precision_test = evaluator_precision.evaluate(pred_test)
    recall_test = evaluator_recall.evaluate(pred_test)
    
    # Log de métricas
    mlflow.log_metric("f1_weighted_train", f1_train)
    mlflow.log_metric("precision_weighted_train", precision_train)
    mlflow.log_metric("recall_weighted_train", recall_train)
    mlflow.log_metric("f1_weighted_test", f1_test)
    mlflow.log_metric("precision_weighted_test", precision_test)
    mlflow.log_metric("recall_weighted_test", recall_test)

    # Log del modelo
    mlflow.spark.log_model(model, "modelo_xgb")

    # Log de parámetros del modelo
    mlflow.log_param("max_depth", xgb.getOrDefault("max_depth"))
    mlflow.log_param("learning_rate", xgb.getOrDefault("learning_rate"))
    mlflow.log_param("n_estimators", xgb.getOrDefault("n_estimators"))
    mlflow.log_param("featuresCol", xgb.getOrDefault("featuresCol"))
    mlflow.log_param("labelCol", xgb.getOrDefault("labelCol"))
#

## 5. Análisis Detallado del Mejor Modelo

In [0]:
model = xgb.fit(train_final_df)

In [0]:
pred_test = model.transform(test_final_df)

In [0]:
# obtener classification_report

# Supón que 'predictions' es el DataFrame con las columnas 'label' y 'prediction'
y_true = pred_test.select("LABEL_ZONE_idx").toPandas()
y_pred = pred_test.select("prediction").toPandas()

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

In [0]:
# Matriz de confusión

# Supón que 'predictions' es tu DataFrame de resultados
y_true = pred_test.select("LABEL_ZONE_idx").toPandas()
y_pred = pred_test.select("prediction").toPandas()

cm = confusion_matrix(y_true, y_pred)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión (Seaborn)')
plt.show()

## 6. Validación Cruzada

In [0]:
paramGrid = ParamGridBuilder().build()

cv = CrossValidator(
    estimator=xgb,  # Ensure xgb is a properly initialized Spark ML estimator compatible with CrossValidator and properly configured
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_f1,
    numFolds=3,
    seed=42
)

cvModel = cv.fit(train_final_df)

In [0]:
# Mostrar metricas de cv
print(cvModel.avgMetrics)

## 7. Registrar Pipeline de transformaciones para usar en produccion

In [0]:
# Registrar pipeline de transformaciones
with mlflow.start_run():
    mlflow.spark.log_model(
        spark_model=selector_model,
        artifact_path="pipeline_transformer",
        registered_model_name="pipe_transformer_registry"
    )

## 8. Registrar Mejor Modelo para usar en produccion

In [0]:
# Datos necesarios
run_id = "1639d0cd200146609bbb823728469a96"  # El ID del experimento/run donde está el modelo
artifact_path = "modelo_xgb"  # El artifact_path usado en mlflow.log_model
model_name = "modelo_xgb_registry"  # Nombre que tendrá en el Model Registry

# Registrar el modelo en el Model Registry
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/{artifact_path}",
    name=model_name
)

print("Modelo registrado en el Model Registry:", result.name, "versión:", result.version)